# Train YOLO26n Detector in Colab

This notebook fine-tunes `yolo26n.pt` on the license plate detector dataset.

It is a clean Colab notebook that:

- mounts Google Drive
- opens the repo from Drive
- installs detector training dependencies
- sanity-checks the current detector dataset
- writes a Colab-safe dataset YAML
- trains `yolo26n.pt` with explicit augmentation settings
- runs test-set evaluation
- saves checkpoints in non-destructive, model-specific locations


## Recommended Baseline

This notebook keeps the detector training settings aligned with the repo baseline and only changes the model size.

- model: `yolo26n.pt`
- epochs: `80`
- image size: `640`
- batch size: `16`
- patience: `20`

Current dataset snapshot expected by this notebook:

- train images: `329`
- val images: `86`
- test images: `85`
- class name: `plate_number`


## Augmentation Choice

This notebook uses a mild augmentation recipe suited to license plate detection on a relatively small one-class dataset.

The goal is to improve robustness without distorting plate appearance too aggressively.

- keep mosaic, but not too strong: useful for a small dataset
- disable left-right flips: mirrored plate text is unrealistic
- disable up-down flips: camera views will not be upside down
- keep small color jitter and moderate scale/translation
- disable mixup and copy-paste: usually not helpful for this task
- disable random erasing: plates are small targets and easy to over-occlude


## Output Safety

This notebook is set up to avoid overwriting your earlier `yolo26s` training outputs.

- training runs go under `outputs/training/usm_lpr_yolo26n/`
- each run name includes a timestamp
- `exist_ok=False` is passed explicitly to Ultralytics
- copied checkpoints go to `models/detector/candidates/`
- this notebook does not overwrite `models/detector/yolo26nbest.pt`
- this notebook does not overwrite the stable `yolo26s` artifacts


In [2]:
from google.colab import drive  # type: ignore[reportMissingImports]

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from datetime import datetime
from pathlib import Path

# Update this if your repo lives in a different Drive folder.
REPO_DIR = Path('/content/drive/MyDrive/plate')

DETECTOR_REQUIREMENTS_PATH = REPO_DIR / 'requirements-detector-colab.txt'
FALLBACK_REQUIREMENTS_PATH = REPO_DIR / 'requirements-colab.txt'
REQUIREMENTS_PATH = DETECTOR_REQUIREMENTS_PATH if DETECTOR_REQUIREMENTS_PATH.exists() else FALLBACK_REQUIREMENTS_PATH

MODEL_NAME = 'yolo26n.pt'
EPOCHS = 200
IMGSZ = 640
BATCH = 16
PATIENCE = 20
SEED = 42

PROJECT = REPO_DIR / 'outputs' / 'training' / 'usm_lpr_yolo26n'
BASE_DATA_YAML = REPO_DIR / 'configs' / 'detector_data.yaml'
DATA_YAML = PROJECT / 'detector_data_colab_yolo26n.yaml'
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'yolo26n_plate_{RUN_STAMP}'

AUGMENTATION = {
    'hsv_h': 0.01,
    'hsv_s': 0.5,
    'hsv_v': 0.3,
    'degrees': 2.0,
    'translate': 0.05,
    'scale': 0.4,
    'shear': 0.0,
    'perspective': 0.0,
    'fliplr': 0.0,
    'flipud': 0.0,
    'mosaic': 0.5,
    'close_mosaic': 10,
    'mixup': 0.0,
    'copy_paste': 0.0,
    'erasing': 0.0,
}

print('REPO_DIR =', REPO_DIR)
print('REQUIREMENTS_PATH =', REQUIREMENTS_PATH)
print('BASE_DATA_YAML =', BASE_DATA_YAML)
print('DATA_YAML =', DATA_YAML)
print('PROJECT =', PROJECT)
print('RUN_NAME =', RUN_NAME)
print('AUGMENTATION =', AUGMENTATION)


REPO_DIR = /content/drive/MyDrive/plate
REQUIREMENTS_PATH = /content/drive/MyDrive/plate/requirements-colab.txt
BASE_DATA_YAML = /content/drive/MyDrive/plate/configs/detector_data.yaml
DATA_YAML = /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/detector_data_colab_yolo26n.yaml
PROJECT = /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n
RUN_NAME = yolo26n_plate_20260420_033728
AUGMENTATION = {'hsv_h': 0.01, 'hsv_s': 0.5, 'hsv_v': 0.3, 'degrees': 2.0, 'translate': 0.05, 'scale': 0.4, 'shear': 0.0, 'perspective': 0.0, 'fliplr': 0.0, 'flipud': 0.0, 'mosaic': 0.5, 'close_mosaic': 10, 'mixup': 0.0, 'copy_paste': 0.0, 'erasing': 0.0}


In [4]:
assert REPO_DIR.exists(), f'Repo directory not found: {REPO_DIR}'
assert REQUIREMENTS_PATH.exists(), f'Requirements file not found: {REQUIREMENTS_PATH}'
assert BASE_DATA_YAML.exists(), f'Detector dataset YAML not found: {BASE_DATA_YAML}'

print('Repo ready:', REPO_DIR)


Repo ready: /content/drive/MyDrive/plate


In [5]:
import os

print('Installing from', REQUIREMENTS_PATH)
os.chdir('/content')
!python -m pip install --upgrade pip
!python -m pip install -r "{REQUIREMENTS_PATH}"


Installing from /content/drive/MyDrive/plate/requirements-colab.txt


In [6]:
import platform
import torch
import ultralytics

print('Python version:', platform.python_version())
print('PyTorch version:', torch.__version__)
print('Ultralytics version:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    print('Current CUDA device index:', device_index)
    print('GPU name:', torch.cuda.get_device_name(device_index))
    free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
    print('GPU memory free (GB):', round(free_bytes / (1024 ** 3), 2))
    print('GPU memory total (GB):', round(total_bytes / (1024 ** 3), 2))
else:
    print('No GPU detected. In Colab, switch Runtime > Change runtime type > T4 GPU or better before training.')


Python version: 3.12.13
PyTorch version: 2.10.0+cu128
Ultralytics version: 8.4.39
CUDA available: True
CUDA device count: 1
Current CUDA device index: 0
GPU name: Tesla T4
GPU memory free (GB): 14.46
GPU memory total (GB): 14.56


In [7]:
data_dir = REPO_DIR / 'data'
images_dir = data_dir / 'images'
labels_dir = data_dir / 'labels'
splits = ('train', 'val', 'test')
image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

expected_counts = {
    'train': 329,
    'val': 86,
    'test': 85,
}

for split in splits:
    split_images = images_dir / split
    split_labels = labels_dir / split
    assert split_images.exists(), f'Missing images directory: {split_images}'
    assert split_labels.exists(), f'Missing labels directory: {split_labels}'

    image_files = sorted([p for p in split_images.iterdir() if p.suffix.lower() in image_suffixes])
    label_files = sorted(split_labels.glob('*.txt'))
    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    print(f'[{split}] images={len(image_files)} labels={len(label_files)} expected_images={expected_counts[split]}')
    print(f'[{split}] missing_label_for_image={len(image_stems - label_stems)} extra_label_without_image={len(label_stems - image_stems)}')

    assert len(image_files) == expected_counts[split], (
        f'Unexpected {split} image count: {len(image_files)} != {expected_counts[split]}'
    )

    malformed_rows = []
    invalid_class_ids = []
    empty_files = []

    for label_path in label_files:
        raw = label_path.read_text(encoding='utf-8').strip()
        if not raw:
            empty_files.append(label_path.name)
            continue

        for line in raw.splitlines():
            parts = line.split()
            if len(parts) != 5:
                malformed_rows.append((label_path.name, line))
                continue
            if parts[0] != '0':
                invalid_class_ids.append((label_path.name, parts[0]))

    print(f'[{split}] empty_label_files={len(empty_files)}')
    if malformed_rows:
        print('  example malformed row:', malformed_rows[0])
    if invalid_class_ids:
        print('  example invalid class id:', invalid_class_ids[0])

    assert not malformed_rows, f'{split} has malformed YOLO rows; example: {malformed_rows[0]}'
    assert not invalid_class_ids, f'{split} has invalid class ids; example: {invalid_class_ids[0]}'


[train] images=329 labels=329 expected_images=329
[train] missing_label_for_image=0 extra_label_without_image=0
[train] empty_label_files=8
[val] images=86 labels=86 expected_images=86
[val] missing_label_for_image=0 extra_label_without_image=0
[val] empty_label_files=0
[test] images=85 labels=85 expected_images=85
[test] missing_label_for_image=0 extra_label_without_image=0
[test] empty_label_files=1


## Note About Empty YOLO Labels

Some empty label files can be intentional Roboflow null-image negatives.

This notebook reports how many there are, but it does not treat them as automatic dataset corruption.


In [8]:
import yaml

PROJECT.mkdir(parents=True, exist_ok=True)

colab_dataset = {
    'path': str(REPO_DIR / 'data'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'plate_number',
    },
}

with DATA_YAML.open('w', encoding='utf-8') as f:
    yaml.safe_dump(colab_dataset, f, sort_keys=False, allow_unicode=False)

print('Wrote dataset YAML to:', DATA_YAML)
print(DATA_YAML.read_text(encoding='utf-8'))


Wrote dataset YAML to: /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/detector_data_colab_yolo26n.yaml
path: /content/drive/MyDrive/plate/data
train: images/train
val: images/val
test: images/test
names:
  0: plate_number



In [9]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(PROJECT),
    name=RUN_NAME,
    patience=PATIENCE,
    seed=SEED,
    exist_ok=False,
    **AUGMENTATION,
)

RUN_DIR = Path(results.save_dir)
print('Training run directory:', RUN_DIR)
results


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/detector_data_colab_yolo26n.yaml, degrees=2.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolo26n_plate_20260420_033728, nbs=64

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78bf83b6ac30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [10]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

test_eval_name = f'{run_dir.name}_test_eval'
best_model = YOLO(str(best_pt))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    project=str(PROJECT),
    name=test_eval_name,
    exist_ok=False,
)

test_metrics.results_dict


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 0.2±0.1 MB/s, size: 104.8 KB)
val: Scanning /content/drive/MyDrive/plate/data/labels/test.cache... 85 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 85/85 25.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 9.6s/it 57.5s2.4ss
                   all         85         97      0.856      0.856      0.885      0.545
Speed: 0.8ms preprocess, 6.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/yolo26n_plate_20260420_033728_test_eval


{'metrics/precision(B)': 0.8560143892375184,
 'metrics/recall(B)': 0.8556701030927835,
 'metrics/mAP50(B)': 0.8852926526810445,
 'metrics/mAP50-95(B)': 0.5447516059943933,
 'fitness': 0.5447516059943933}

In [11]:
from pathlib import Path
import shutil

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

target_dir = REPO_DIR / 'models' / 'detector' / 'candidates'
target_dir.mkdir(parents=True, exist_ok=True)

target_best = target_dir / f'{run_dir.name}_best.pt'
target_last = target_dir / f'{run_dir.name}_last.pt'

assert not target_best.exists(), f'Target best checkpoint already exists: {target_best}'
shutil.copy2(best_pt, target_best)
print('Copied best checkpoint to:', target_best)

if last_pt.exists():
    assert not target_last.exists(), f'Target last checkpoint already exists: {target_last}'
    shutil.copy2(last_pt, target_last)
    print('Copied last checkpoint to:', target_last)

print('Stable detector alias left untouched:', REPO_DIR / 'models' / 'detector' / 'yolo26nbest.pt')


Copied best checkpoint to: /content/drive/MyDrive/plate/models/detector/candidates/yolo26n_plate_20260420_033728_best.pt
Copied last checkpoint to: /content/drive/MyDrive/plate/models/detector/candidates/yolo26n_plate_20260420_033728_last.pt
Stable detector alias left untouched: /content/drive/MyDrive/plate/models/detector/best.pt


In [12]:
from pathlib import Path

run_dir = Path(results.save_dir) if 'results' in globals() else None
if run_dir is None or not run_dir.exists():
    candidate_dirs = sorted(
        [path for path in PROJECT.glob(f'{RUN_NAME}*') if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    assert candidate_dirs, f'No run directories found under {PROJECT} matching {RUN_NAME}*'
    run_dir = candidate_dirs[-1]

test_eval_dir = PROJECT / f'{run_dir.name}_test_eval'
candidate_dir = REPO_DIR / 'models' / 'detector' / 'candidates'
print('Run directory:', run_dir)
print('Test evaluation directory:', test_eval_dir)
print('Candidate checkpoint directory:', candidate_dir)
print('Weights directory contents:')
for path in sorted((run_dir / 'weights').glob('*')):
    print('-', path)


Run directory: /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/yolo26n_plate_20260420_033728
Test evaluation directory: /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/yolo26n_plate_20260420_033728_test_eval
Candidate checkpoint directory: /content/drive/MyDrive/plate/models/detector/candidates
Weights directory contents:
- /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/yolo26n_plate_20260420_033728/weights/best.pt
- /content/drive/MyDrive/plate/outputs/training/usm_lpr_yolo26n/yolo26n_plate_20260420_033728/weights/last.pt


## Optional Promotion Step

This notebook intentionally does not replace `models/detector/yolo26nbest.pt`.

After you compare `yolo26n` against your existing `yolo26s` results, you can manually promote the chosen checkpoint later if it is actually better.


## After Training

When this run finishes:

1. review validation metrics and training plots
2. review test-set metrics from the evaluation cell
3. compare the `yolo26n` results against your `yolo26s` baseline
4. keep the preferred checkpoint in `models/detector/`
5. continue with OCR and end-to-end evaluation using the rest of the repo workflow
